# Import library

In [161]:
import os
import gc
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import WeightedRandomSampler
from tqdm import tqdm
from tabulate import tabulate
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

In [162]:
# =========================================================
# 0) BASIC SETUP
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


# 1) Dataset

In [163]:
import re
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path


class EEGDataset(Dataset):
    def __init__(
        self,
        npz_path: str,
        mtf_path: str,
        normalize: bool = True,
        image_size: int = 128
    ):
        data = np.load(npz_path)
        self.X = data["X"].astype(np.float32)   # (N, 18, 1024)
        self.y = data["y"].astype(np.int64)     # (N,)

        self.normalize = normalize
        self.image_size = image_size

        assert len(self.X) == len(self.y), "X và y không khớp"

        # -------------------------------------------------
        # Build mapping: idx -> image_path
        # ảnh được đặt tên theo format: image_{win:05d}.jpg
        # -------------------------------------------------
        image_files = list(Path(mtf_path).rglob("image_*.jpg"))
        pattern = re.compile(r"image_(\d{5})\.jpg$")

        idx_to_path = {}
        for p in image_files:
            match = pattern.search(p.name)
            if match is None:
                continue

            win_idx = int(match.group(1))

            if win_idx in idx_to_path:
                raise ValueError(
                    f"Trùng ảnh cho cùng idx={win_idx}:\n"
                    f"  {idx_to_path[win_idx]}\n"
                    f"  {p}"
                )

            idx_to_path[win_idx] = str(p)

        self.idx_to_path = idx_to_path

        # -------------------------------------------------
        # Check thiếu / thừa ảnh
        # -------------------------------------------------
        n_samples = len(self.X)
        missing_idxs = [i for i in range(n_samples) if i not in self.idx_to_path]
        extra_idxs = [i for i in self.idx_to_path.keys() if i >= n_samples]

        if missing_idxs:
            preview = missing_idxs[:10]
            raise ValueError(
                f"Thiếu {len(missing_idxs)} ảnh không khớp với X. "
                f"Ví dụ idx thiếu: {preview}"
            )

        if extra_idxs:
            preview = extra_idxs[:10]
            raise ValueError(
                f"Có {len(extra_idxs)} ảnh thừa so với số sample trong X. "
                f"Ví dụ idx thừa: {preview}"
            )

    def __len__(self):
        return len(self.X)

    @staticmethod
    def minmax_to_minus1_1(x: np.ndarray, axis=None, eps: float = 1e-8):
        x_min = x.min(axis=axis, keepdims=True)
        x_max = x.max(axis=axis, keepdims=True)
        x = 2.0 * (x - x_min) / (x_max - x_min + eps) - 1.0
        return x

    def __getitem__(self, idx):
        # -------- signal --------
        x = self.X[idx]   # (18, 1024)

        if self.normalize:
            # Chuẩn hoá từng channel riêng về [-1, 1]
            # axis=1 vì mỗi channel có 1024 điểm thời gian
            x = self.minmax_to_minus1_1(x, axis=1)

        x = torch.from_numpy(x).float()
        y = torch.tensor(self.y[idx], dtype=torch.long)

        # -------- image --------
        img_path = self.idx_to_path[idx]
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Không đọc được ảnh: {img_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size))

        # [0,255] -> [0,1] -> [-1,1]
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        img = img * 2.0 - 1.0

        return {
            "signal": x,      # (18, 1024), trong [-1, 1]
            "image": img,     # (3, image_size, image_size), trong [-1, 1]
            "label": y
        }

# 2) SIGNAL MODEL: TRANSFORMER

In [164]:
import torch
import torch.nn as nn


class EEGSignalTransformer(nn.Module):
    def __init__(
        self,
        in_channels: int = 18,
        seq_len: int = 2048,
        patch_size: int = 32,
        conv_channels=64,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 256,
        dropout: float = 0.1
    ):
        super().__init__()
        assert seq_len % patch_size == 0, "seq_len phải chia hết cho patch_size"
        assert d_model % nhead == 0, "d_model phải chia hết cho nhead"

        self.in_channels = in_channels
        self.seq_len = seq_len
        self.patch_size = patch_size
        self.num_patches = seq_len // patch_size
        self.conv_channels = conv_channels
        self.d_model = d_model

        # =====================================================
        # 1) Conv frontend để học đặc trưng cục bộ theo thời gian
        # Input:  (B, C, T)
        # Output: (B, conv_channels, T)
        # =====================================================
        self.frontend = nn.Sequential(
            nn.Conv1d(in_channels, conv_channels, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # =====================================================
        # 2) Patch embedding
        # Mỗi patch có shape: (conv_channels, patch_size)
        # flatten -> Linear -> d_model
        # =====================================================
        self.patch_dim = conv_channels * patch_size
        self.patch_embed = nn.Linear(self.patch_dim, d_model)

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # Positional embedding cho CLS + patch tokens
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, d_model))

        self.embed_dropout = nn.Dropout(dropout)

        # =====================================================
        # 3) Transformer encoder
        # =====================================================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(d_model)

        # output feature dimension
        self.out_dim = d_model

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        if isinstance(self.patch_embed, nn.Linear):
            nn.init.xavier_uniform_(self.patch_embed.weight)
            if self.patch_embed.bias is not None:
                nn.init.zeros_(self.patch_embed.bias)

    def patchify(self, x):
        """
        x: (B, conv_channels, T)
        return: (B, num_patches, conv_channels * patch_size)
        """
        B, C, T = x.shape
        assert T == self.seq_len, f"Expected seq_len={self.seq_len}, got {T}"

        x = x.view(B, C, self.num_patches, self.patch_size)   # (B, C, N, P)
        x = x.permute(0, 2, 1, 3).contiguous()                # (B, N, C, P)
        x = x.view(B, self.num_patches, self.patch_dim)       # (B, N, C*P)
        return x

    def forward(self, x):
        """
        x: (B, 18, 2048)
        return: (B, d_model)
        """
        # 1) local feature extraction
        x = self.frontend(x)                                  # (B, conv_channels, T)

        # 2) patchify + embedding
        x = self.patchify(x)                                  # (B, N, patch_dim)
        x = self.patch_embed(x)                               # (B, N, d_model)

        # 3) thêm CLS token
        B = x.size(0)
        cls_tokens = self.cls_token.expand(B, 1, self.d_model)  # (B, 1, d_model)
        x = torch.cat([cls_tokens, x], dim=1)                   # (B, N+1, d_model)

        # 4) positional embedding
        x = x + self.pos_embed
        x = self.embed_dropout(x)

        # 5) transformer encoder
        x = self.encoder(x)
        x = self.norm(x)

        # 6) lấy CLS token làm feature cuối
        feat = x[:, 0]                                        # (B, d_model)
        return feat

# 3) IMAGE MODEL: MOBILENET

In [165]:
class MobileNetImageEncoder(nn.Module):
    def __init__(self, pretrained=True, out_dim=128):
        super().__init__()

        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        backbone = mobilenet_v3_small(weights=weights)

        self.features = backbone.features
        self.avgpool = backbone.avgpool

        # mobilenet_v3_small -> sau avgpool + flatten ra 576 chiều
        self.proj = nn.Linear(576, out_dim)
        self.out_dim = out_dim

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)   # (B, 576)
        x = self.proj(x)          # (B, out_dim)
        return x

# 4) FUSION MODEL

In [166]:
class FusionEEGNet(nn.Module):
    def __init__(
        self,
        signal_model,
        image_model,
        signal_dim=128,
        image_dim=128,
        num_classes=2
    ):
        super().__init__()
        self.signal_model = signal_model
        self.image_model = image_model

        # head cho signal branch
        self.signal_head = nn.Sequential(
            nn.Linear(signal_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes)
        )

        # head cho image branch
        self.image_head = nn.Sequential(
            nn.Linear(image_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes)
        )

        # head cho fusion branch
        self.fusion_head = nn.Sequential(
            nn.Linear(signal_dim + image_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, signal, image):
        feat_signal = self.signal_model(signal)   # [B, 128]
        feat_image = self.image_model(image)      # [B, 128]

        logits_signal = self.signal_head(feat_signal)
        logits_image = self.image_head(feat_image)

        feat_fusion = torch.cat([feat_signal, feat_image], dim=1)
        logits_fusion = self.fusion_head(feat_fusion)

        return logits_signal, logits_image, logits_fusion

# 5) TRAIN / VALIDATE

In [167]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    w_signal=0.2,
    w_image=0.2,
    w_fusion=0.6
):
    model.train()

    running_total_loss = 0.0
    running_signal_loss = 0.0
    running_image_loss = 0.0
    running_fusion_loss = 0.0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc="Train", leave=False)

    for batch in pbar:
        signal = batch["signal"].to(device, non_blocking=True)
        image = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits_signal, logits_image, logits_fusion = model(signal, image)

        loss_signal = criterion(logits_signal, label)
        loss_image = criterion(logits_image, label)
        loss_fusion = criterion(logits_fusion, label)

        total_loss = (
            w_signal * loss_signal +
            w_image * loss_image +
            w_fusion * loss_fusion
        )

        total_loss.backward()
        optimizer.step()

        bs = signal.size(0)

        running_total_loss += total_loss.item() * bs
        running_signal_loss += loss_signal.item() * bs
        running_image_loss += loss_image.item() * bs
        running_fusion_loss += loss_fusion.item() * bs

        preds = torch.argmax(logits_fusion, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(label.detach().cpu().numpy())

        pbar.set_postfix(
            total=f"{total_loss.item():.4f}",
            sig=f"{loss_signal.item():.4f}",
            img=f"{loss_image.item():.4f}",
            fus=f"{loss_fusion.item():.4f}"
        )

    epoch_total_loss = running_total_loss / len(loader.dataset)
    epoch_signal_loss = running_signal_loss / len(loader.dataset)
    epoch_image_loss = running_image_loss / len(loader.dataset)
    epoch_fusion_loss = running_fusion_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return {
        "loss": epoch_total_loss,
        "loss_signal": epoch_signal_loss,
        "loss_image": epoch_image_loss,
        "loss_fusion": epoch_fusion_loss,
        "acc": acc,
        "bal_acc": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [168]:
@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device,
    w_signal=0.2,
    w_image=0.2,
    w_fusion=0.6
):
    model.eval()

    running_total_loss = 0.0
    running_signal_loss = 0.0
    running_image_loss = 0.0
    running_fusion_loss = 0.0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc="Val", leave=False)

    for batch in pbar:
        signal = batch["signal"].to(device, non_blocking=True)
        image = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        logits_signal, logits_image, logits_fusion = model(signal, image)

        loss_signal = criterion(logits_signal, label)
        loss_image = criterion(logits_image, label)
        loss_fusion = criterion(logits_fusion, label)

        total_loss = (
            w_signal * loss_signal +
            w_image * loss_image +
            w_fusion * loss_fusion
        )

        bs = label.size(0)

        running_total_loss += total_loss.item() * bs
        running_signal_loss += loss_signal.item() * bs
        running_image_loss += loss_image.item() * bs
        running_fusion_loss += loss_fusion.item() * bs

        preds = torch.argmax(logits_fusion, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(label.cpu().numpy().tolist())

        pbar.set_postfix(
            total=f"{total_loss.item():.4f}",
            sig=f"{loss_signal.item():.4f}",
            img=f"{loss_image.item():.4f}",
            fus=f"{loss_fusion.item():.4f}"
        )

    epoch_total_loss = running_total_loss / len(loader.dataset)
    epoch_signal_loss = running_signal_loss / len(loader.dataset)
    epoch_image_loss = running_image_loss / len(loader.dataset)
    epoch_fusion_loss = running_fusion_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )

    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, zero_division=0)

    return {
        "loss": epoch_total_loss,
        "loss_signal": epoch_signal_loss,
        "loss_image": epoch_image_loss,
        "loss_fusion": epoch_fusion_loss,
        "acc": acc,
        "bal_acc": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "cm": cm,
        "report": report
    }

# 6) PATHS

In [169]:
train_wavelet_dir = "/kaggle/input/datasets/hnguyentt/chb12-loso-30mins/chb12/fold_test_sid_6/train_wavelet_gray_custom_band"
val_wavelet_dir   = "/kaggle/input/datasets/hnguyentt/chb12-loso-30mins/chb12/fold_test_sid_6/validation_wavelet_gray_custom_band"
test_wavelet_dir = "/kaggle/input/datasets/hnguyentt/chb12-loso-30mins/chb12/fold_test_sid_6/test_wavelet_gray_custom_band"

train_npz = "/kaggle/input/datasets/hnguyentt/chb12-loso-30mins/chb12/fold_test_sid_6/train.npz"
val_npz   = "/kaggle/input/datasets/hnguyentt/chb12-loso-30mins/chb12/fold_test_sid_6/validation.npz"
test_npz  = "/kaggle/input/datasets/hnguyentt/chb12-loso-30mins/chb12/fold_test_sid_6/test.npz"

# 7) DATASET / DATALOADER

In [170]:
train_dataset = EEGDataset(
    npz_path=train_npz,
    mtf_path=train_wavelet_dir,
    normalize=True,
    image_size=224
)

val_dataset = EEGDataset(
    npz_path=val_npz,
    mtf_path=val_wavelet_dir,
    normalize=True,
    image_size=224
)

test_dataset = EEGDataset(
    npz_path=test_npz,
    mtf_path=test_wavelet_dir,
    normalize=True,
    image_size=224
)


train_labels = train_dataset.y
class_counts = np.bincount(train_labels)
print("Train class counts:", class_counts)

Train class counts: [2867 2867]


In [171]:
from matplotlib import pyplot as plt

In [172]:
# from torch.utils.data import Dataset, WeightedRandomSampler
# class_weights = 1.0 / class_counts
# sample_weights = class_weights[train_labels]
# sample_weights = torch.from_numpy(sample_weights).double()

# sampler = WeightedRandomSampler(
#     weights=sample_weights,
#     num_samples=len(sample_weights),
#     replacement=True
# )

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=512,
#     sampler=sampler,
#     num_workers=2,
#     pin_memory=True,
#     drop_last=True,
# )


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)


val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 8) MODEL

In [173]:
signal_model = EEGSignalTransformer(
    in_channels=18,
    seq_len=2048,
    patch_size=8,
    conv_channels=256,
    d_model=256,
    nhead=8,
    num_layers=3,
    dim_feedforward=256,
    dropout=0.2
)

image_model = MobileNetImageEncoder(
    pretrained=True,
    out_dim=256
)

model = FusionEEGNet(
    signal_model=signal_model,
    image_model=image_model,
    signal_dim=256,
    image_dim=256,
    num_classes=2
).to(device)

/tmp/ipykernel_55/275565210.py:72: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


# 9) CHECK SHAPE 1 BATCH

In [174]:
batch = next(iter(train_loader))
signal = batch["signal"].to(device)
image = batch["image"].to(device)

with torch.no_grad():
    logits_signal, logits_image, logits_fusion = model(signal, image)

print("logits_signal:", logits_signal.shape)
print("logits_image :", logits_image.shape)
print("logits_fusion:", logits_fusion.shape)

logits_signal: torch.Size([64, 2])
logits_image : torch.Size([64, 2])
logits_fusion: torch.Size([64, 2])


# 10) LOSS / OPTIMIZER

In [175]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# 11) TRAIN LOOp

In [176]:
# =========================================================
# 11) TRAIN LOOP
# =========================================================
num_epochs = 30
best_val_f1 = -1.0

# trọng số 3 loss
w_signal = 0.3
w_image = 0.2
w_fusion = 0.5

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        w_signal=w_signal,
        w_image=w_image,
        w_fusion=w_fusion
    )

    val_metrics = validate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        w_signal=w_signal,
        w_image=w_image,
        w_fusion=w_fusion
    )

    table = [
        [
            "Train",
            f"{train_metrics['loss']:.4f}",
            f"{train_metrics['loss_signal']:.4f}",
            f"{train_metrics['loss_image']:.4f}",
            f"{train_metrics['loss_fusion']:.4f}",
            f"{train_metrics['acc']:.4f}",
            f"{train_metrics['bal_acc']:.4f}",
            f"{train_metrics['precision']:.4f}",
            f"{train_metrics['recall']:.4f}",
            f"{train_metrics['f1']:.4f}",
        ],
        [
            "Val",
            f"{val_metrics['loss']:.4f}",
            f"{val_metrics['loss_signal']:.4f}",
            f"{val_metrics['loss_image']:.4f}",
            f"{val_metrics['loss_fusion']:.4f}",
            f"{val_metrics['acc']:.4f}",
            f"{val_metrics['bal_acc']:.4f}",
            f"{val_metrics['precision']:.4f}",
            f"{val_metrics['recall']:.4f}",
            f"{val_metrics['f1']:.4f}",
        ],
    ]

    print(tabulate(
        table,
        headers=[
            "Split", "Total Loss", "Signal Loss", "Image Loss", "Fusion Loss",
            "Acc", "Bal Acc", "Precision", "Recall", "F1"
        ],
        tablefmt="grid"
    ))

    print("Val confusion matrix:")
    print(val_metrics["cm"])
    print("Val classification report:")
    print(val_metrics["report"])

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(model.state_dict(), "best_fusion_eegnet_new_30m_shuffle.pth")
        print("Saved best model.")


Epoch [1/30]


+---------+--------------+---------------+--------------+---------------+-------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |   Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+=======+===========+=============+==========+========+
| Train   |       0.6478 |        0.659  |        0.624 |        0.6507 | 0.599 |     0.599 |      0.6131 |    0.599 | 0.5861 |
+---------+--------------+---------------+--------------+---------------+-------+-----------+-------------+----------+--------+
| Val     |       0.557  |        0.4863 |        0.743 |        0.5249 | 0.681 |     0.681 |      0.8053 |    0.681 | 0.6449 |
+---------+--------------+---------------+--------------+---------------+-------+-----------+-------------+----------+--------+
Val confusion matrix:
[[21 37]
 [ 0 58]]
Val classification report:
              precision    recall  f

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.5192 |        0.604  |       0.4817 |        0.4833 | 0.7556 |    0.7556 |      0.7637 |   0.7556 | 0.7537 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       1.0887 |        0.2631 |       1.4711 |        1.4311 | 0.5    |    0.5    |      0.25   |   0.5    | 0.3333 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[ 0 58]
 [ 0 58]]
Val classification report:
              precision    re

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.3824 |        0.5584 |       0.3196 |        0.3018 | 0.8708 |    0.8707 |      0.8718 |   0.8707 | 0.8707 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       1.7691 |        0.5704 |       2.0036 |        2.3945 | 0.5    |    0.5    |      0.25   |   0.5    | 0.3333 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[ 0 58]
 [ 0 58]]
Val classification report:
              precision    re

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.2953 |        0.5366 |       0.2054 |        0.1864 | 0.9245 |    0.9245 |      0.9247 |   0.9245 | 0.9245 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       2.1327 |        0.358  |       3.1921 |        2.7739 | 0.5    |    0.5    |      0.25   |   0.5    | 0.3333 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[ 0 58]
 [ 0 58]]
Val classification report:
              precision    re

...

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.0236 |        0.0746 |       0.0044 |        0.0007 | 0.9998 |    0.9998 |      0.9998 |   0.9998 | 0.9998 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       0.6732 |        0.465  |       0.9391 |        0.6918 | 0.8966 |    0.8966 |      0.9087 |   0.8966 | 0.8958 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[47 11]
 [ 1 57]]
Val classification report:
              precision    re

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.0247 |        0.0776 |       0.0049 |        0.0009 | 0.9998 |    0.9998 |      0.9998 |   0.9998 | 0.9998 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       0.2673 |        0.1251 |       0.5645 |        0.2337 | 0.9397 |    0.9397 |      0.9462 |   0.9397 | 0.9394 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[51  7]
 [ 0 58]]
Val classification report:
              precision    re

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.0359 |        0.1065 |       0.0107 |        0.0037 | 0.9991 |    0.9991 |      0.9991 |   0.9991 | 0.9991 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       0.4797 |        0.1615 |       0.964  |        0.4769 | 0.931  |    0.931  |      0.9357 |   0.931  | 0.9308 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[51  7]
 [ 1 57]]
Val classification report:
              precision    re

+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+========+===========+=============+==========+========+
| Train   |       0.0233 |        0.0636 |       0.013  |        0.0033 | 0.9989 |    0.9989 |      0.9989 |   0.9989 | 0.9989 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
| Val     |       0.9631 |        0.4174 |       1.4561 |        1.0933 | 0.819  |    0.819  |      0.8671 |   0.819  | 0.8128 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+-------------+----------+--------+
Val confusion matrix:
[[37 21]
 [ 0 58]]
Val classification report:
              precision    re

+---------+--------------+---------------+--------------+---------------+-------+-----------+-------------+----------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |   Acc |   Bal Acc |   Precision |   Recall |     F1 |
+=========+==============+===============+==============+===============+=======+===========+=============+==========+========+
| Train   |       0.025  |        0.0778 |       0.0073 |        0.0005 | 1     |     1     |      1      |    1     | 1      |
+---------+--------------+---------------+--------------+---------------+-------+-----------+-------------+----------+--------+
| Val     |       0.4378 |        0.4006 |       0.6686 |        0.3677 | 0.931 |     0.931 |      0.9394 |    0.931 | 0.9307 |
+---------+--------------+---------------+--------------+---------------+-------+-----------+-------------+----------+--------+
Val confusion matrix:
[[50  8]
 [ 0 58]]
Val classification report:
              precision    recall  f

# 12 Evaluate

In [177]:
def load_checkpoint_for_test(model, ckpt_path, device, strict=True):
    ckpt = torch.load(ckpt_path, map_location=device)

    # Một số file save trực tiếp state_dict
    # Một số file save dạng {"model_state_dict": ...}
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
        else:
            state_dict = ckpt
    else:
        state_dict = ckpt

    # Nếu model được save bằng DataParallel
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict, strict=strict)
    model.to(device)
    model.eval()

    print(f"Đã load weight từ: {ckpt_path}")
    return model

In [178]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)


@torch.no_grad()
def predict_test(
    model,
    loader,
    criterion,
    device,
    w_signal=0.2,
    w_image=0.2,
    w_fusion=0.6,
):
    model.eval()

    running_total_loss = 0.0
    running_signal_loss = 0.0
    running_image_loss = 0.0
    running_fusion_loss = 0.0
    total_samples = 0

    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(loader, desc="Test", leave=False)

    for batch in pbar:
        signal = batch["signal"].to(device, non_blocking=True)
        image = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        logits_signal, logits_image, logits_fusion = model(signal, image)

        loss_signal = criterion(logits_signal, label)
        loss_image = criterion(logits_image, label)
        loss_fusion = criterion(logits_fusion, label)

        total_loss = (
            w_signal * loss_signal +
            w_image * loss_image +
            w_fusion * loss_fusion
        )

        bs = label.size(0)
        total_samples += bs

        running_total_loss += total_loss.item() * bs
        running_signal_loss += loss_signal.item() * bs
        running_image_loss += loss_image.item() * bs
        running_fusion_loss += loss_fusion.item() * bs

        probs = torch.softmax(logits_fusion, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(label.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

        pbar.set_postfix(
            total=f"{total_loss.item():.4f}",
            sig=f"{loss_signal.item():.4f}",
            img=f"{loss_image.item():.4f}",
            fus=f"{loss_fusion.item():.4f}",
        )

    if total_samples == 0:
        raise ValueError("Test loader không có sample nào.")

    epoch_total_loss = running_total_loss / total_samples
    epoch_signal_loss = running_signal_loss / total_samples
    epoch_image_loss = running_image_loss / total_samples
    epoch_fusion_loss = running_fusion_loss / total_samples

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )

    cm = confusion_matrix(all_labels, all_preds)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        # fallback cho multi-class (macro average)
        sensitivity = recall  # đã là macro recall
        specificity = np.nan  # không định nghĩa rõ cho multi-class
    
    report = classification_report(
        all_labels,
        all_preds,
        zero_division=0,
        digits=4
    )

    results = {
        "loss": epoch_total_loss,
        "loss_signal": epoch_signal_loss,
        "loss_image": epoch_image_loss,
        "loss_fusion": epoch_fusion_loss,
        "acc": acc,
        "bal_acc": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "cm": cm,
        "report": report,
        "preds": np.array(all_preds),
        "labels": np.array(all_labels),
        "probs": np.array(all_probs),
    }

    return results

In [179]:
from tabulate import tabulate


def print_test_results(test_results):
    table = [
        [
            "Test",
            f"{test_results['loss']:.4f}",
            f"{test_results['loss_signal']:.4f}",
            f"{test_results['loss_image']:.4f}",
            f"{test_results['loss_fusion']:.4f}",
            f"{test_results['acc']:.4f}",
            f"{test_results['bal_acc']:.4f}",
            f"{test_results['precision']:.4f}",
            f"{test_results['recall']:.4f}",
            f"{test_results['f1']:.4f}",
            f"{test_results['sensitivity']:.4f}",
            f"{test_results['specificity']:.4f}",
        ]
    ]
    
    headers = [
        "Split",
        "Total Loss",
        "Signal Loss",
        "Image Loss",
        "Fusion Loss",
        "Acc",
        "Bal Acc",
        "Pre",
        "Recall",
        "F1",
        "Sen",
        "Spe",
    ]

    print(tabulate(table, headers=headers, tablefmt="grid"))
    print("Test confusion matrix:")
    print(test_results["cm"])
    print("Test classification report:")
    print(test_results["report"])

In [180]:
ckpt_path = "/kaggle/working/best_fusion_eegnet_new_30m_shuffle.pth"   # đường dẫn file weight của bạn

model = load_checkpoint_for_test(
    model=model,
    ckpt_path=ckpt_path,
    device=device,
    strict=True
)

test_results = predict_test(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    w_signal=0.2,
    w_image=0.3,
    w_fusion=0.5
)

print_test_results(test_results)

Đã load weight từ: /kaggle/working/best_fusion_eegnet_new_30m_shuffle.pth


+---------+--------------+---------------+--------------+---------------+--------+-----------+--------+----------+--------+--------+--------+
| Split   |   Total Loss |   Signal Loss |   Image Loss |   Fusion Loss |    Acc |   Bal Acc |    Pre |   Recall |     F1 |    Sen |    Spe |
+=========+==============+===============+==============+===============+========+===========+========+==========+========+========+========+
| Test    |       1.0199 |        0.2334 |       1.5626 |        1.0088 | 0.8492 |    0.8492 | 0.8536 |   0.8492 | 0.8487 | 0.9048 | 0.7937 |
+---------+--------------+---------------+--------------+---------------+--------+-----------+--------+----------+--------+--------+--------+
Test confusion matrix:
[[50 13]
 [ 6 57]]
Test classification report:
              precision    recall  f1-score   support

           0     0.8929    0.7937    0.8403        63
           1     0.8143    0.9048    0.8571        63

    accuracy                         0.8492       126
  